# Balance Checks & McCray Test

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdrobust import rdrobust
from rdrobust import rdbwselect
from rdrobust import rdplot
from rddensity import rddensity, rdplotdensity
import re

In [2]:
df_rdd = pd.read_csv('df_diffindisc_clean.csv')

In [3]:
df_rdd

,code_commune,election_year,nuance,is_far_right,tour,voix,treat,d_50,skill_share_pre,skill_share_trend,...,delta_net_rate_all,delta_in_rate_no_bac,delta_out_rate_no_bac,delta_net_rate_no_bac,delta_in_rate_bac_plus,delta_out_rate_bac_plus,delta_net_rate_bac_plus,delta_in_rate_tertiary,delta_out_rate_tertiary,delta_net_rate_tertiary
0,1053,2014,LUG,False,t1,6515,0,-39.704625,0.392705,0.020687,...,-0.009553,-0.011679,-0.000540,-0.011138,-0.014168,-0.008534,-0.005634,-0.003384,-0.007593,0.004209
1,1053,2020,LUG,False,t1,4892,0,-52.243829,0.428149,0.031427,...,0.011807,-0.002043,-0.005401,0.003358,0.006531,-0.015876,0.022407,0.005016,-0.022085,0.027101
2,2168,2014,LDVG,False,t1,3000,0,-46.400496,0.345043,0.057388,...,0.015813,-0.004473,-0.022898,0.018424,-0.001122,-0.011620,0.010498,-0.010501,-0.007550,-0.002951
3,2168,2020,LDVC,False,t1,2434,0,-56.405136,0.342097,-0.013547,...,-0.002961,-0.002715,-0.012601,0.009886,-0.016079,0.003553,-0.019632,-0.014364,0.018223,-0.032587
4,2173,2014,LDVD,False,t2,2246,0,-32.943809,0.300207,0.059974,...,-0.010228,-0.014428,0.000423,-0.014851,-0.015190,-0.015633,0.000443,-0.026910,-0.020112,-0.006798
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
832,95491,2014,LDVD,False,t1,2122,0,-50.989660,0.523774,0.058217,...,0.019801,0.010774,-0.008805,0.019578,0.010069,-0.009040,0.019108,-0.048612,-0.010673,-0.037939
833,95563,2014,LUMP,False,t1,3393,0,-43.121279,0.550017,0.032736,...,-0.000970,0.010042,0.019654,-0.009612,0.011793,0.006937,0.004856,-0.000661,0.005619,-0.006279
834,95607,2008,LUG,False,t1,4889,0,-46.320816,0.430999,0.062273,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
835,95607,2014,LUMP,False,t2,5224,0,-41.834061,0.496955,0.041926,...,0.009312,-0.004874,-0.006150,0.001276,0.011326,-0.005415,0.016741,0.008433,-0.003008,0.011442


## Select Bandwith

- **h (the estimation window)**: the communes used to estimate the jump. It’s narrow, only compare close races.
- **b (the bias window)**: a wider window used to check how much the straight-line fit bends near the cutoff and to correct for that bend. The robust p-value and CI rely on this correction.

In [4]:
bw_by_cycle, b_by_cycle, bw_results = {}, {}, {}
y_col = "delta_net_rate_tertiary"  # main outcome: change in net migration of tertiary graduates

sample = df_rdd[df_rdd["election_year"].isin([2014, 2020])].dropna(subset=[y_col, "d_50"])
pooled = rdbwselect(y=sample[y_col], x=sample["d_50"], c=0, bwselect="mserd")

for cycle in [2014, 2020]:
    sub = sample[sample["election_year"] == cycle]
    try:
        bw_results[cycle] = rdbwselect(y=sub[y_col], x=sub["d_50"], c=0, bwselect="mserd")
    except Exception as e:
        print(f"{cycle}: own bandwidth failed ({e}), using pooled")
        bw_results[cycle] = pooled

# h = estimation bandwidth, b = bias-correction bandwidth
bw_by_cycle = {cycle: float(res.bws.iloc[0, 0]) for cycle, res in bw_results.items()}
b_by_cycle = {cycle: float(res.bws.iloc[0, 2]) for cycle, res in bw_results.items()}

print("h:", bw_by_cycle)
print("b:", b_by_cycle)

2020: own bandwidth failed (4-th leading minor of the array is not positive definite), using pooled
h: {2014: 5.337572379425224, 2020: 27.639116507721848}
b: {2014: 9.76836517966954, 2020: 39.67300079230032}


## Balance Checks

In [5]:
# Check d_50 range within each election cycle
for y in [2008, 2014, 2020]:
    sub = df_rdd[df_rdd['election_year'] == y]['d_50']
    print(f"{y}: min={sub.min():.2f}, max={sub.max():.2f}, n_treated={sub[sub>0].count()}, n_control={sub[sub<=0].count()}")

2008: min=-75.00, max=-20.48, n_treated=0, n_control=39
2014: min=-90.65, max=48.26, n_treated=13, n_control=500
2020: min=-88.33, max=55.99, n_treated=13, n_control=272


#### Data Limitation
- **No treated units in year 2008**
So I add try and except statement to skipp 2008 if necessary

Note on 2008: No commune in the 2008 sample has d_50 > 0, meaning the far-right won a plurality in none of the observed elections that year. Since we require observations on both sides of the cutoff to fit two local regressions, the cutoff c=0 falls outside the empirical support of d_50 and estimation is not possible. We therefore exclude 2008 from all balance checks and from the main Diff-in-Disc estimation, and report this as a sample limitation.

In [6]:
def balance_check(df, covariate, running='d_50', cutoff=0, cycle=None, h=None):
    """Run rdrobust for one covariate, optionally restricted to one election
    cycle, using fixed bandwidths h (estimation) and b (bias correction) so
    that all balance tests within a cycle use the same windows as the
    main outcome regression."""
    b = b_by_cycle.get(cycle)   # MSE-optimal bias bandwidth of this cycle (None for 2008 -> skipped anyway)

    data = df.dropna(subset=[covariate, running])
    if cycle is not None:
        data = data[data['election_year'] == cycle]

    try:
        result = rdrobust(y=data[covariate], x=data[running], c=cutoff, h=h, b=b) # sharp RDD on one cross-section per cycle
    except Exception as e:
        print(f"Skipped {covariate} ({cycle}): {e}")
        return None

    return {
        'covariate': covariate,                                # name of the covariate
        'cycle': cycle if cycle is not None else 'pooled',      # election year (2008/2014/2020)
        'bw_used': h,                                       # estimation bandwidth h actually used
        'b_used': b,                                        # bias-correction bandwidth b actually used
        'coef': result.coef.loc['Robust'].values[0],        # estimated discontinuity (jump) in the covariate at the cutoff
        'se': result.se.loc['Robust'].values[0],            # standard error of that estimate (robust, bias-corrected)
        'pval': result.pv.loc['Robust'].values[0],          # p-value — probability of seeing a jump this large if there were truly no discontinuity
        'ci_lower': result.ci.loc['Robust'].values[0],      # lower bound of the 95% confidence interval for the jump
        'ci_upper': result.ci.loc['Robust'].values[1],      # upper bound of the 95% confidence interval for the jump
        'N_total': sum(result.N),                           # full sample available (left+right)
        'N_eff': sum(result.N_h),                           # effective N actually used within bw — the honest number to report
    }

In [7]:
covariates = ['log_pop', 'skill_share_pre', 'skill_share_trend', 'gender_gap_tertiary',
              'no_diploma_share', 'income_median', 'income_gini', 'unemployment_rate',
              'pop_growth', 'foreign_population', 'share_55plus', 'share_15_24']

In [8]:
df_rdd

,code_commune,election_year,nuance,is_far_right,tour,voix,treat,d_50,skill_share_pre,skill_share_trend,...,delta_net_rate_all,delta_in_rate_no_bac,delta_out_rate_no_bac,delta_net_rate_no_bac,delta_in_rate_bac_plus,delta_out_rate_bac_plus,delta_net_rate_bac_plus,delta_in_rate_tertiary,delta_out_rate_tertiary,delta_net_rate_tertiary
0,1053,2014,LUG,False,t1,6515,0,-39.704625,0.392705,0.020687,...,-0.009553,-0.011679,-0.000540,-0.011138,-0.014168,-0.008534,-0.005634,-0.003384,-0.007593,0.004209
1,1053,2020,LUG,False,t1,4892,0,-52.243829,0.428149,0.031427,...,0.011807,-0.002043,-0.005401,0.003358,0.006531,-0.015876,0.022407,0.005016,-0.022085,0.027101
2,2168,2014,LDVG,False,t1,3000,0,-46.400496,0.345043,0.057388,...,0.015813,-0.004473,-0.022898,0.018424,-0.001122,-0.011620,0.010498,-0.010501,-0.007550,-0.002951
3,2168,2020,LDVC,False,t1,2434,0,-56.405136,0.342097,-0.013547,...,-0.002961,-0.002715,-0.012601,0.009886,-0.016079,0.003553,-0.019632,-0.014364,0.018223,-0.032587
4,2173,2014,LDVD,False,t2,2246,0,-32.943809,0.300207,0.059974,...,-0.010228,-0.014428,0.000423,-0.014851,-0.015190,-0.015633,0.000443,-0.026910,-0.020112,-0.006798
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
832,95491,2014,LDVD,False,t1,2122,0,-50.989660,0.523774,0.058217,...,0.019801,0.010774,-0.008805,0.019578,0.010069,-0.009040,0.019108,-0.048612,-0.010673,-0.037939
833,95563,2014,LUMP,False,t1,3393,0,-43.121279,0.550017,0.032736,...,-0.000970,0.010042,0.019654,-0.009612,0.011793,0.006937,0.004856,-0.000661,0.005619,-0.006279
834,95607,2008,LUG,False,t1,4889,0,-46.320816,0.430999,0.062273,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
835,95607,2014,LUMP,False,t2,5224,0,-41.834061,0.496955,0.041926,...,0.009312,-0.004874,-0.006150,0.001276,0.011326,-0.005415,0.016741,0.008433,-0.003008,0.011442


In [9]:
# 1. Log population
df_rdd['log_pop'] = np.log(df_rdd['pop_pre'])
results = [balance_check(df_rdd, 'log_pop', cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]
results = [r for r in results if r is not None]

Skipped log_pop (2008): c should be set within the range of x


**Log population**: Population size proxies for a commune's general character (urban/rural, economic base) and directly affects how d_50 behaves in plurality elections with few voters. If treated and control communes differ systematically in size, any downstream effect could just reflect "small vs. large towns," not treatment.

In [10]:
# 2. Pre-treatment skill share (level)
results += [balance_check(df_rdd, 'skill_share_pre', cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]

Skipped skill_share_pre (2008): c should be set within the range of x


**Skill share, pre-treatment (level)**: This is the value of your outcome variable before the election. If it already differs at the cutoff, treated and control communes weren't comparable to begin with, which directly threatens the causal interpretation of delta_skill_share.

In [11]:
# 3. Pre-treatment skill share trend
results += [balance_check(df_rdd, 'skill_share_trend', cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]

Skipped skill_share_trend (2008): c should be set within the range of x


**Skill share trend**: Checks whether communes were already moving in different directions in educated-population share before the election, not just sitting at different starting levels. A jump here would mean your Diff-in-Disc design needs the trend, not just the level, netted out.

In [12]:
# 4. Gender gap in tertiary education
results += [balance_check(df_rdd, 'gender_gap_tertiary', cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]

Skipped gender_gap_tertiary (2008): c should be set within the range of x


**Gender gap in tertiary education**: Captures whether the balance of highly-educated men vs. women differs across the cutoff, which can reflect underlying differences in local economic structure (e.g. university towns, specific industries) rather than random assignment.

In [13]:
# 5. No-diploma share
results += [balance_check(df_rdd, 'no_diploma_share', cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]

Skipped no_diploma_share (2008): c should be set within the range of x


**No-diploma share**: The mirror image of skill share, capturing the low-education end of the distribution. Checking it separately from skill share confirms there isn't a compositional imbalance skill share alone might miss.

In [14]:
# 6. Income median
results += [balance_check(df_rdd, 'income_median', cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]

Skipped income_median (2008): c should be set within the range of x


**Income median**: A textbook confounder: income plausibly predicts both far-right vote share (economic-anxiety channel) and future skill composition (ability of high earners to move). If it's imbalanced, it's a live threat to your causal claim.

In [15]:
# 7. Income Gini
results += [balance_check(df_rdd, 'income_gini', cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]

Skipped income_gini (2008): c should be set within the range of x


**Income Gini**: Captures inequality within a commune, separately from its average income level — two communes can have the same median but very different equality, and inequality independently correlates with political polarization.

In [16]:
# 8. Unemployment rate
results += [balance_check(df_rdd, 'unemployment_rate', cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]

Skipped unemployment_rate (2008): c should be set within the range of x


**Unemployment rate**: The single most theoretically motivated confounder given the well-documented far-right/economic-distress link worldwide; if unemployed communes disproportionately elect the far-right and separately drive skilled outmigration, this variable sits directly on your identification's weakest point.

In [17]:
# 9. Population growth (log change over the pre-election census interval)
results += [balance_check(df_rdd, 'pop_growth', cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]

Skipped pop_growth (2008): zero-size array to reduction operation minimum which has no identity


**Population growth**: Fast-growing communes (often peri-urban, with many new arrivals) differ from shrinking ones, and newcomers change the education mix of a commune on their own. If treated communes were already growing faster before the election, a change in skill share could just reflect who moved in, not the far-right win.

In [18]:
# 10. Foreign-born share (% of population, pre-election census)
results += [balance_check(df_rdd, 'foreign_population', cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]

Skipped foreign_population (2008): zero-size array to reduction operation minimum which has no identity


**Foreign-born share**: A classic ingredient in the far-right debate: the share of immigrants in a commune may relate both to how people vote and to who lives there and what education they have. If treated and control communes differ here, the comparison mixes the election result with the local population mix.

In [19]:
# 11. Age structure (% aged 55+ and % aged 15-24, pre-election census)
results += [balance_check(df_rdd, 'share_55plus', cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]
results += [balance_check(df_rdd, 'share_15_24',  cycle=y, h=bw_by_cycle.get(y)) for y in [2008, 2014, 2020]]

Skipped share_55plus (2008): zero-size array to reduction operation minimum which has no identity
Skipped share_15_24 (2008): zero-size array to reduction operation minimum which has no identity


**Age structure (55+ and 15-24)**: Older communes (many retirees) and young communes (students, young workers) differ in both voting habits and education levels, since older cohorts hold fewer diplomas. If treated communes are older or younger at the cutoff, changes in skill share could simply reflect ageing or student turnover.

In [20]:
# ── Assemble final balance table ──
results = [r for r in results if r is not None]  # drop skipped (2008, empty-treatment) checks
balance_table = pd.DataFrame(results)
balance_table

,covariate,cycle,bw_used,b_used,coef,se,pval,ci_lower,ci_upper,N_total,N_eff
0,log_pop,2014,5.337572,9.768365,-0.449889,0.748228,0.547658,-1.916389,1.016610,513,7
1,log_pop,2020,27.639117,39.673001,2.130344,1.180963,0.071246,-0.184301,4.444989,285,38
2,skill_share_pre,2014,5.337572,9.768365,-0.047572,0.038092,0.211721,-0.122231,0.027088,513,7
3,skill_share_pre,2020,27.639117,39.673001,0.013481,0.128097,0.916187,-0.237585,0.264547,285,38
4,skill_share_trend,2014,5.337572,9.768365,0.007645,0.017782,0.667245,-0.027206,0.042496,513,7
5,skill_share_trend,2020,27.639117,39.673001,0.001937,0.017413,0.911446,-0.032193,0.036066,285,38
6,gender_gap_tertiary,2014,5.337572,9.768365,-0.006146,0.027263,0.821631,-0.059581,0.047288,513,7
7,gender_gap_tertiary,2020,27.639117,39.673001,0.110879,0.060524,0.066956,-0.007747,0.229504,285,38
8,no_diploma_share,2014,5.337572,9.768365,0.047572,0.038092,0.211721,-0.027088,0.122231,513,7
9,no_diploma_share,2020,27.639117,39.673001,-0.013481,0.128097,0.916187,-0.264547,0.237585,285,38


**H0 = no discontinuity** (*(β = 0) → you want to fail to reject*)
- We want do detect nothing, hence a discontinuity in a pre-treatment covariates is evidance against "as-good-as-random" assignment.

**p < 0.05 = reject H0** → discontinuity detected → bad for identification

### [NO UPDATE] Potential Problems

| Covariate | Cycle | p-value |
|---|---|---|
| `log_pop` | 2014 | 0.032 |
| `skill_share_pre` | 2014 | 0.047 |
| `no_diploma_share` | 2014 | 0.047 *(mirror of `skill_share_pre`)* |
| `unemployment_rate` | 2014 | 0.027 |
| `gender_gap_tertiary` | 2020 | 0.001 |

## McCrary Test

In [23]:
# ── McCrary-style manipulation test on the running variable ──
# rddensity implements the Cattaneo, Jansson & Ma (2018/2020) bias-corrected
# local-polynomial density test — the modern, standard-error-robust version
# of the original McCrary (2008) test.


# Run separately by cycle (2014, 2020): manipulation incentives, list registration
# rules, and vote-counting differ by election, so pooling would obscure a
# discontinuity that's real in only one cycle. 2008 is excluded — no observations
# above the cutoff means there is nothing to test for a jump against.
for cycle in [2014, 2020]:
    sub = df_rdd.loc[df_rdd['election_year'] == cycle, 'd_50'].dropna()

    dens = rddensity(X=sub, c=0)  # c=0: cutoff of d_50, as in the main design

    # p_jk = jackknife p-value for H0: density is continuous at the cutoff.
    # p_jk > 0.10 (conventional threshold) => no evidence of manipulation/sorting.
    p_val = dens.test.loc['p_jk']
    print(f"McCrary density test ({cycle}): p = {p_val:.3f} "
          f"({'no evidence of manipulation' if p_val > 0.10 else 'possible manipulation — investigate'})")

    # Visual check: plots the estimated density on each side of the cutoff.
    # A visible jump at d_50 = 0 is the graphical analogue of a low p-value.
    rdplotdensity(dens, sub, title=f"Density of d_50 — {cycle}")

McCrary density test (2014): p = 0.834 (no evidence of manipulation)
McCrary density test (2020): p = 0.883 (no evidence of manipulation)


Density test fails in both election periods:

- **Mechanical asymmetry**: close far-right wins are just intrinsically rarer than close losses in this setting (e.g., because far-right lists are rarely the largest list, so barely winning the plurality is a rarer event than barely losing it) — this is a structural feature of French multi-list plurality elections, not manipulation.
- **True local sorting**: something specifically happens right at d_50=0 that doesn't happen at d_50=-0.3 or +0.3 — more consistent with actual manipulation (vote-count adjustment, contested results, etc.).


As Diff-in-Disc does not require this sharp RDD randomization at the cutoff. 

In [24]:
# ── Diagnose asymmetry driving a failed density test ──
# Rather than tweaking bandwidth/threshold to force p > 0.05 (specification
# searching), report the raw near-cutoff counts: this tells you whether the
# failure is "narrow wins are just rare" (mechanical) vs. a sharp local jump
# right at zero (more consistent with sorting).
for cycle in [2014, 2020]:
    sub = df_rdd.loc[df_rdd['election_year'] == cycle, 'd_50'].dropna()
    n_left  = ((sub > -0.5) & (sub < 0)).sum()   # narrow losses
    n_right = ((sub >= 0)   & (sub < 0.5)).sum() # narrow wins
    print(f"{cycle}: narrow losses (-0.5,0)={n_left}, narrow wins [0,0.5)={n_right}, "
          f"total treated={(sub > 0).sum()}, total control={(sub <= 0).sum()}")

# ── Placebo density test: same test, run on the running variable of the
# PRIOR election cycle for the same communes (t-1 placebo). If the
# discontinuity/asymmetry already existed before treatment could have acted,
# it's a stable structural feature of the setting -- consistent with the
# Diff-in-Disc identifying assumption (Grembi, Nannicini & Troiano 2016) --
# rather than manipulation triggered by treatment itself.
# NOTE: requires a lagged d_50 column (e.g. 'd_50_lag') merged in beforehand.
if 'd_50_lag' in df_rdd.columns:
    for cycle in [2014, 2020]:
        sub_lag = df_rdd.loc[df_rdd['election_year'] == cycle, 'd_50_lag'].dropna()
        dens_lag = rddensity(X=sub_lag, c=0)
        p_lag = dens_lag.test.loc['p_jk']
        print(f"Placebo density test, t-1 margin ({cycle}): p = {p_lag:.3f}")

2014: narrow losses (-0.5,0)=1, narrow wins [0,0.5)=1, total treated=13, total control=500
2020: narrow losses (-0.5,0)=0, narrow wins [0,0.5)=0, total treated=13, total control=272
